# ?? ???? ?? ????


### ??? ????


In [ ]:
import os
from pprint import pprint

import optuna
import numpy as np
import pandas as pd
#import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
import pandas as pd

THRESHOLD = 0.3
RANDOM_STATE = 110

train_data = pd.read_csv("./data/final_train_data_ver2.csv")
test_data = pd.read_csv("./data/final_test_data_ver2.csv")

In [ ]:
# dam, fill1, fill2 ?? ??
var_dam_fill = [
    'Equipment_same_num',
    'PalletID_Collect_Result_encoded',
    'Production_Qty_Collect_Result_encoded',
    'WorkMode Collect Result',
    'Receip_n_suffix_3',
    'time_gap_All'
]

In [ ]:
# ?? ?? ??
### correlation ??? ?? ?? ???
var_all_corr = [
    'model_suffix_encoded',
    'cleaned_workorder_encoded'
]

### train
var_all_train = [
    'target',
    'model_suffix_encoded',
    'cleaned_workorder_encoded'
]

### test
var_all_test = [
    'Set ID',
    'target',
    'model_suffix_encoded',
    'cleaned_workorder_encoded'
]

In [ ]:
# '_Dam'? ???? ?? ??
dam_variables = [var for var in train_data.columns if '_Dam' in var]

# train
final_columns_train = var_dam_fill + var_all_train + dam_variables
train_data_dam = train_data[final_columns_train]

# test 
final_columns_test = var_dam_fill + var_all_test + dam_variables
test_data_dam = test_data[final_columns_test]

In [ ]:
# '_Fill1'? ???? ?? ??
fill1_variables = [var for var in train_data.columns if '_Fill1' in var]

# train
final_columns_train = var_dam_fill + var_all_train + fill1_variables
train_data_fill1 = train_data[final_columns_train]

# test 
final_columns_test = var_dam_fill + var_all_test + fill1_variables
test_data_fill1 = test_data[final_columns_test]

In [ ]:
# '_Fill2'? ???? ?? ??
fill2_variables = [var for var in train_data.columns if '_Fill2' in var]

# train
final_columns_train = var_dam_fill + var_all_train + fill2_variables
train_data_fill2 = train_data[final_columns_train]

# test 
final_columns_test = var_dam_fill + var_all_test + fill2_variables
test_data_fill2 = test_data[final_columns_test]

In [ ]:
# '_AutoClave'? ???? ?? ??
autoclave_variables = [var for var in train_data.columns if '_AutoClave' in var]

# train
final_columns_train = var_all_train + autoclave_variables
train_data_autoclave = train_data[final_columns_train]

# test 
final_columns_test = var_all_test + autoclave_variables
test_data_autoclave = test_data[final_columns_test]

In [ ]:
# ? DataFrame? ?? ? ??
num_columns_train_data = train_data.shape[1]
num_columns_train_data_dam = train_data_dam.shape[1]
num_columns_train_data_autoclave = train_data_autoclave.shape[1]
num_columns_train_data_fill1 = train_data_fill1.shape[1]
num_columns_train_data_fill2 = train_data_fill2.shape[1]

num_columns_test_data = test_data.shape[1]
num_columns_test_data_dam = test_data_dam.shape[1]
num_columns_test_data_autoclave = test_data_autoclave.shape[1]
num_columns_test_data_fill1 = test_data_fill1.shape[1]
num_columns_test_data_fill2 = test_data_fill2.shape[1]

# ? DataFrame? ?? ? ??
print("----train data-----")
print(f"train_data DataFrame? ?? ?: {num_columns_train_data}")
print(f"train_data_dam DataFrame? ?? ?: {num_columns_train_data_dam}")
print(f"train_data_autoclave DataFrame? ?? ?: {num_columns_train_data_autoclave}")
print(f"train_data_fill1 DataFrame? ?? ?: {num_columns_train_data_fill1}")
print(f"train_data_fill2 DataFrame? ?? ?: {num_columns_train_data_fill2}")
print("----test data-----")
print(f"test_data DataFrame? ?? ?: {num_columns_test_data}")
print(f"test_data_dam DataFrame? ?? ?: {num_columns_test_data_dam}")
print(f"test_data_autoclave DataFrame? ?? ?: {num_columns_test_data_autoclave}")
print(f"test_data_fill1 DataFrame? ?? ?: {num_columns_test_data_fill1}")
print(f"test_data_fill2 DataFrame? ?? ?: {num_columns_test_data_fill2}")

---

## Optuna

????? 0.3?? ?????? ?? ??? ?

In [ ]:
RANDOM_STATE = 110
THRESHOLD = 0.3

In [ ]:
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

def objectiveCatBoost(trial, x_tr, y_tr, x_val, y_val):
    
    # 'Normal'? 'AbNormal'? ??? ??
    y_tr = y_tr.map({'Normal': 0, 'AbNormal': 1})
    y_val = y_val.map({'Normal': 0, 'AbNormal': 1})
    
    param = {
        'iterations': trial.suggest_int('iterations', 800, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.2),
        'depth': trial.suggest_int('depth', 4, 13),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 5),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True),
        'random_strength': trial.suggest_float('random_strength', 0, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.1, 10),
        'border_count': trial.suggest_int('border_count', 128, 300),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 10),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
        
        'random_seed': RANDOM_STATE,
        'eval_metric': 'F1',
        'logging_level': 'Silent',
        'boosting_type': 'Plain'
    }

    model = CatBoostClassifier(**param)
    model.fit(x_tr, y_tr)
    pred_proba = model.predict_proba(x_val)[:, 1]  # ?? ??? ??
    pred = (pred_proba >= THRESHOLD).astype(int)  # ?????? ?? ??
    
    score = f1_score(y_val, pred, average="binary")
    
    return score

# ???? ??
x_train, x_val, y_train, y_val = train_test_split(
    train_data_autoclave.drop("target", axis=1),  
    train_data_autoclave["target"],               
    test_size=0.2,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# ??? ???? ??
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(lambda trial: objectiveCatBoost(trial, x_train, y_train, x_val, y_val), n_trials=400)

print('Best trial: score {}, \nparams {}'.format(study.best_trial.value, study.best_trial.params))

1. 0.24528301886792453

params {'iterations': 2453, 'learning_rate': 0.045101812655324314, 'depth': 7, 'l2_leaf_reg': 2.7766650741754, 'random_strength': 7.514157578279006, 'bagging_temperature': 7.736554866068651, 'border_count': 301, 'scale_pos_weight': 1.73893896002959, 'grow_policy': 'SymmetricTree'}

In [ ]:
# ?? ?? ? ???????
models = {
    'et': ExtraTreesClassifier(),
    'rf': RandomForestClassifier(),
    'cat': CatBoostClassifier(),
    'lgbm': LGBMClassifier(),
    'xgb': XGBClassifier(),
    'dt': DecisionTreeClassifier()
}

def train_and_evaluate_model(model_name, data, **params):
    if model_name not in models:
        print(f"{model_name}?(?) ???? ?? ?????.")
        return
    
    # ???? ??
    x_train, x_val, y_train, y_val = train_test_split(
        data.drop("target", axis=1),
        data["target"].map({'Normal': 0, 'AbNormal': 1}),
        test_size=0.2,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    # ?? ??
    model = models[model_name]

    # ??????? ??
    model.set_params(**params)

    # ?? ??
    model.fit(x_train, y_train)

    # ??? ??? ???? ???? ?? ?? ??
    data_name = [name for name in globals() if globals()[name] is data][0]

    # ??
    y_val_pred_proba = model.predict_proba(x_val)[:, 1]  # ?? ??? ??
    y_val_pred = (y_val_pred_proba >= THRESHOLD).astype(int)  # ?????? ?? ??

    # ???? ??
    f1 = f1_score(y_val, y_val_pred, average="binary")
    accuracy = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # ???? ??
    f1 = f1_score(y_val, y_val_pred, average="binary")
    accuracy = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # ?? ??
    print(f'{model_name} ??? {data_name} ???? ??? ??:')
    print(f'F1 Score: {f1}')
    print('---')
    print('Confusion Matrix:')
    print(conf_matrix)
    print('---')
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print('\n')

In [ ]:
train_and_evaluate_model(
    'cat', train_data_autoclave,
    iterations = 2786, 
    learning_rate = 0.016342560305036093, 
    depth = 8, 
    min_data_in_leaf = ,
    l2_leaf_reg = 3.7187150890684246, 
    random_strength = 0.13164684607188099, 
    bagging_temperature = 9.823498597792092, 
    border_count = 158, 
    scale_pos_weight = 1.8735070170496537,
    grow_policy = 'SymmetricTree',

    random_state = RANDOM_STATE,
    eval_metric = 'F1',
    logging_level = 'Silent',
    boosting_type = 'Plain'
)